# Domain conservation across isoforms -> the Figure 1C-F master table

Projects every Pfam domain defined on a gene's reference isoform (MANE Select / Ensembl canonical) onto the gene's other protein-coding isoforms, scores retention (intact / partially trimmed / skipped, and reading-frame preservation), and writes a per-domain **master table** (`domain_conservation_master.tsv`) annotated with Pfam clan and CDS-exon count. That table is the input from which the manuscript's **Figure 1C-F** panels are plotted (the final figures are composed separately).

In [1]:
# Force the inline backend — under `jupyter nbconvert --execute` the
# default sometimes lands on Agg, which prints `<Figure …>` instead of
# the actual PNG. The magic call forces module://matplotlib_inline.backend_inline.
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except NameError:
    pass  # Not in IPython (e.g. plain python REPL); ignore.
import matplotlib as mpl
import matplotlib.pyplot as plt

# Every figure saved as PNG also gets a vector PDF (same bbox/dimensions) under
# reproduce_paper/figures/ (right next to these notebooks), so the
# paper figures are available as scalable PDFs without changing any plotting code.
from pathlib import Path as _Path
import os as _os
FIGDIR = (_Path(_os.environ.get("FASTCDS_REPO") or (_Path.home() / "Desktop" / "fastCDS"))
          / "reproduce_paper" / "figures")
FIGDIR.mkdir(parents=True, exist_ok=True)
if not getattr(mpl.figure.Figure.savefig, "_writes_pdf", False):  # guard: don't double-wrap
    _orig_savefig = mpl.figure.Figure.savefig
    def _savefig_both(self, fname, *a, **k):
        _orig_savefig(self, fname, *a, **k)
        s = str(fname)
        if s.lower().endswith(".png"):
            kk = dict(k); kk.pop("dpi", None)              # vector PDF: drop raster dpi
            _orig_savefig(self, str(FIGDIR / (_os.path.splitext(_os.path.basename(s))[0] + ".pdf")), *a, **kk)
    _savefig_both._writes_pdf = True
    mpl.figure.Figure.savefig = _savefig_both

# Paper-ready figure defaults. Tweaks vs matplotlib's stock style:
#   - Larger fonts (10pt body, 11pt axis labels, 12pt title).
#   - Thinner spines + only-left/-bottom by default (less chartjunk).
#   - Subtle horizontal grid; no vertical grid.
#   - tab10 palette but used sparingly — we override per-plot.
plt.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 200,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'axes.titleweight': 'semibold',
    'axes.titlepad': 10,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.linewidth':    0.8,
    'axes.grid': True,
    'axes.grid.axis': 'y',
    'grid.color': '#e5e7eb',
    'grid.linewidth': 0.8,
    'xtick.major.size': 4,
    'ytick.major.size': 4,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'legend.frameon': False,
    'legend.fontsize': 10,
    'lines.linewidth': 2.0,
})

# Colorblind-safe palette (Wong 2011, also used in seaborn's 'colorblind').
COLORS = {
    'fastCDS':   '#0072B2',  # blue
    'ensembldb':   '#009E73',  # bluish green
    'transvar':    '#E69F00',  # orange
    'rest':        '#CC79A7',  # reddish-purple
    'good':        '#009E73',
    'bad':         '#D55E00',  # vermilion (works for colorblind)
    'neutral':     '#56B4E9',
    'highlight':   '#F0E442',
}

import os, sys, subprocess
from pathlib import Path
import pandas as pd
import numpy as np
import fastCDS as fc

# Importing fastCDS flips matplotlib to the Agg backend; re-assert inline so
# figures embed in the notebook under `jupyter nbconvert --execute`.
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

REPO  = _Path.home() / "Desktop" / "fastCDS"
DATA  = _Path.home() / "Desktop" / "protein2genomic_data"
VAR   = _Path.home() / "Desktop" / "tfregdb2" / "data" / "variants"
BENCH = REPO / "reproduce_paper" / "benchmarks"
CONS  = REPO / "reproduce_paper" / "conservation"
CONS.mkdir(parents=True, exist_ok=True)
IDX   = DATA / "zenodo_indices" / "ensembl_v115_human.idx"
GTF   = DATA / "Homo_sapiens.GRCh38.115.gtf"
PY    = sys.executable

def step(out, cmd):
    """Run a prep command only if its output is missing (idempotent)."""
    out = _Path(out)
    if out.exists():
        print("cached:", out.name); return out
    print("running:", out.name, "...")
    subprocess.run([str(c) for c in cmd], check=True)
    return out

print("v115 index present:", IDX.exists())

v115 index present: True


## Stage 0 — isoform universe

Parse the release-115 GTF into one row per protein-coding isoform (gene,
transcript, protein, MANE/canonical flags, CDS length, exon count) and keep
genes with **≥ 2** coding isoforms. The reference ("source") isoform per gene is
MANE Select if present, else Ensembl canonical, else the longest CDS.

In [2]:
# Build the v115 index once (matches Pfam v115 protein IDs); ~20 s if missing.
if not IDX.exists():
    fc.build_index(str(GTF), out=str(IDX))

step(CONS / "isoforms.tsv", [
    PY, BENCH / "build_isoform_universe.py", "--gtf", GTF,
    "--out-isoforms", CONS / "isoforms.tsv", "--out-genes", CONS / "genes.tsv"])

genes = pd.read_csv(CONS / "genes.tsv", sep="\t")
isos  = pd.read_csv(CONS / "isoforms.tsv", sep="\t", dtype={"chrom": str})
print(f"multi-isoform genes:      {len(genes):,}")
print(f"protein-coding isoforms:  {len(isos):,}")
print(f"median isoforms per gene: {int(genes['n_isoforms'].median())}")
print(genes['source_basis'].value_counts().to_string())

cached: isoforms.tsv


multi-isoform genes:      16,910
protein-coding isoforms:  208,615
median isoforms per gene: 8
source_basis
mane_select          16739
ensembl_canonical      170
longest_cds              1


## Stages 1–3 — project every source domain and score conservation

`isoform_conservation.py` selects every Pfam domain that sits on a gene's source
isoform, projects it with fastCDS, maps every other isoform structure-only,
and computes `coverage` / `inframe_coverage` per `(domain, target)` pair. (Run
once; cached afterwards.)

In [3]:
step(CONS / "conservation_long.tsv", [
    PY, BENCH / "isoform_conservation.py",
    "--isoforms", CONS / "isoforms.tsv", "--genes", CONS / "genes.tsv",
    "--pfam-meta", DATA / "pfam_human_v115_meta.tsv",
    "--index", IDX, "--out", CONS / "conservation_long.tsv", "--threads", "6"])

con = pd.read_csv(CONS / "conservation_long.tsv", sep="\t")
print(f"(domain x target) pairs: {len(con):,}")
print((con["conservation_class"].value_counts(normalize=True) * 100).round(1).to_string())

# Domain-level conservation score + metadata join
atlas = pd.read_csv(DATA / "atlas_v115_by_domain.tsv", sep="\t",
                    usecols=["query_id", "domain_length_aa", "is_single_exon_domain",
                             "fclass", "protein_length_aa"])
g = con.groupby("query_id")
dom = pd.DataFrame({
    "n_targets": g.size(),
    "frac_conserved": g["conservation_class"].apply(lambda s: (s == "conserved").mean()),
    "mean_inframe": g["inframe_coverage"].mean(),
    "gene_name": g["gene_name"].first(),
    "pfam_id": g["pfam_id"].first(),
}).reset_index().merge(atlas, on="query_id", how="left")
print(f"domains scored: {len(dom):,}   median frac_conserved: {dom['frac_conserved'].median():.2f}")

cached: conservation_long.tsv


(domain x target) pairs: 505,361
conservation_class
conserved    75.8
lost         19.5
partial       4.6


domains scored: 43,601   median frac_conserved: 0.75


## The Figure 1C-F master table

In [4]:
# ===== Master table for Figure 1C-F =====
# One row per source-isoform Pfam domain, carrying everything needed to plot the
# manuscript's Figure 1C-F:
#   1C retention pie      -> n_intact / n_trimmed / n_skipped (summed over all domains)
#   1D intactness bars    -> frac_intact (fraction of a gene's isoforms that keep the domain)
#   1E clan intactness    -> group by clan, distribution of frac_intact
#   1F clan exon topology -> group by clan, distribution of n_exons_spanned (1..5+)
g = con.groupby("query_id")
master = pd.DataFrame({
    "n_targets":   g.size(),
    "n_intact":    g["conservation_class"].apply(lambda s: (s == "conserved").sum()),
    "n_trimmed":   g["conservation_class"].apply(lambda s: (s == "partial").sum()),
    "n_skipped":   g["conservation_class"].apply(lambda s: (s == "lost").sum()),
    "frac_intact": g["conservation_class"].apply(lambda s: (s == "conserved").mean()),
    "mean_inframe_coverage": g["inframe_coverage"].mean(),  # continuous score, for re-thresholding
    "gene_name":   g["gene_name"].first(),
    "pfam_id":     g["pfam_id"].first(),
}).reset_index()

# CDS-exon count + domain size from the Pfam atlas
_atlas = pd.read_csv(DATA / "atlas_v115_by_domain.tsv", sep="\t",
                     usecols=["query_id", "n_touched", "n_cds_exons",
                              "domain_length_aa", "is_single_exon_domain"])
master = (master.merge(_atlas, on="query_id", how="left")
                .rename(columns={"n_touched": "n_exons_spanned"}))

# Pfam clan (families with no clan form their own singleton group)
_clans = (pd.read_csv(DATA / "Pfam-A.clans.tsv", sep="\t", header=None,
                      names=["acc", "clan", "clan_name", "name", "desc"])
            .drop_duplicates("acc").set_index("acc"))
master["clan"]      = master["pfam_id"].map(_clans["clan"]).fillna(master["pfam_id"])
master["clan_name"] = master["pfam_id"].map(_clans["clan_name"])

out = CONS / "domain_conservation_master.tsv"
master.to_csv(out, sep="\t", index=False)
tot = master[["n_intact", "n_trimmed", "n_skipped"]].sum()
print(f"wrote {out}")
print(f"  {len(master):,} domains x {master.shape[1]} columns; "
      f"{(master['clan'] != master['pfam_id']).sum():,} in a real clan across "
      f"{master.loc[master['clan'] != master['pfam_id'], 'clan'].nunique()} clans")
print(f"  overall intact / trimmed / skipped (domain x isoform pairs): "
      f"{100*tot['n_intact']/tot.sum():.1f} / {100*tot['n_trimmed']/tot.sum():.1f} / "
      f"{100*tot['n_skipped']/tot.sum():.1f} %")
master.head()


wrote /home/goguxor/Desktop/fastCDS/reproduce_paper/conservation/domain_conservation_master.tsv
  43,601 domains x 15 columns; 33,734 in a real clan across 557 clans
  overall intact / trimmed / skipped (domain x isoform pairs): 75.8 / 4.6 / 19.5 %


,query_id,n_targets,n_intact,n_trimmed,n_skipped,frac_intact,mean_inframe_coverage,gene_name,pfam_id,n_exons_spanned,n_cds_exons,is_single_exon_domain,domain_length_aa,clan,clan_name
0,PFAM0000000,5,4,1,0,0.800000,0.870600,ARF5,PF00025,6.0,6.0,False,170.0,CL0023,P-loop_NTPase
1,PFAM0000001,18,10,2,6,0.555556,0.655367,M6PR,PF02157,6.0,6.0,False,254.0,CL0226,M6PR
2,PFAM0000002,32,29,0,3,0.906250,0.921800,ESRRA,PF00105,2.0,6.0,False,69.0,CL0839,Tbcl_zf
3,PFAM0000003,32,25,0,7,0.781250,0.850675,ESRRA,PF00104,3.0,6.0,False,170.0,PF00104,NaN
4,PFAM0000004,15,9,1,5,0.600000,0.631507,FKBP4,PF00254,3.0,10.0,False,91.0,CL0487,FKBP


## Verification

Two guards on the engine:

1. **Self-projection** — a domain projected onto its *own* source isoform must
   score exactly `coverage = inframe_coverage = 1.0`.
2. **TP53 DBD** — the DNA-binding domain is p53's constitutive structural core,
   so it should be conserved across most TP53 isoforms.

In [5]:
sys.path.insert(0, str(BENCH))
import isoform_conservation as ic

mp = fc.Mapper(str(IDX), threads=4)
src = mp.map_batch([dict(id="ENSP00000269305", aa_start=95, aa_end=289,
                         domain_id="TP53_DBD")], output="coding")
seg = src.cds_segments; seg = seg[seg.overlaps_domain == "coding_overlap"]
ddom = ic._segments(seg)["TP53_DBD"]
st = ic._structures(mp.map_batch([dict(id="ENSP00000269305", domain_id="ENSP00000269305")],
                                 output="isoform").isoform)["ENSP00000269305"]
bp, cov, inf = ic._score(ddom, st)
assert abs(cov/bp - 1.0) < 1e-9 and abs(inf/bp - 1.0) < 1e-9, "self-projection must be 1.0"
print(f"[1] self-projection TP53 DBD: coverage={cov/bp:.3f} inframe={inf/bp:.3f}  OK")

tp = dom[dom.gene_name == "TP53"][["query_id","pfam_id","frac_conserved","n_targets"]]
print("[2] TP53 Pfam domains across isoforms:")
print(tp.to_string(index=False))

[1] self-projection TP53 DBD: coverage=1.000 inframe=1.000  OK
[2] TP53 Pfam domains across isoforms:
   query_id pfam_id  frac_conserved  n_targets
PFAM0010850 PF08563        0.606061         33
PFAM0010851 PF18521        0.757576         33
PFAM0010852 PF00870        0.909091         33
PFAM0010853 PF07710        0.666667         33


## Summary - what `domain_conservation_master.tsv` contains

One row per source-isoform Pfam domain. Key columns for Figure 1C-F:

| column | meaning | used by |
|---|---|---|
| `n_intact` / `n_trimmed` / `n_skipped` | # of the gene's isoforms in which the domain is intact / partially trimmed / skipped | **1C** (pie) |
| `frac_intact` | fraction of the gene's isoforms that keep the domain fully intact | **1D**, **1E** |
| `mean_inframe_coverage` | continuous retention score (for re-thresholding intact/trimmed/skipped) | 1C-E |
| `clan` / `clan_name` | Pfam clan of the domain (singleton = own family) | **1E**, **1F** |
| `n_exons_spanned` (`n_touched`) | number of CDS exons the domain spans | **1F** |
| `n_cds_exons`, `domain_length_aa`, `is_single_exon_domain` | extra per-domain metadata | - |

The per-`(domain x isoform)` long table (`conservation_long.tsv`) is also on disk if you need the pairwise continuous scores rather than the per-domain aggregate.